In [1]:
import pandas as pd
import numpy as np
import matplotlib as plt
import zipfile

zf = zipfile.ZipFile('COVID-19_Case_Surveillance_Public_Use_Data_with_Geography_20260203.csv.zip')
covid_data = pd.read_csv(zf.open('COVID-19_Case_Surveillance_Public_Use_Data_with_Geography_20260203.csv'), low_memory=False)
covid_data

,case_month,res_state,state_fips_code,res_county,county_fips_code,age_group,sex,race,ethnicity,case_positive_specimen_interval,case_onset_interval,process,exposure_yn,current_status,symptom_status,hosp_yn,icu_yn,death_yn,underlying_conditions_yn
0,2022-07,MO,29,DUNKLIN,29069.0,18 to 49 years,Female,Black,NaN,0.0,0.0,Missing,Missing,Laboratory-confirmed case,Symptomatic,Unknown,Missing,Unknown,NaN
1,2022-03,MO,29,LINCOLN,29113.0,18 to 49 years,Female,NaN,NaN,0.0,NaN,Missing,Missing,Laboratory-confirmed case,Missing,Unknown,Missing,Unknown,NaN
2,2021-09,MO,29,SALINE,29195.0,0 - 17 years,Female,NaN,NaN,0.0,NaN,Missing,Missing,Probable Case,Missing,Unknown,Missing,Unknown,NaN
3,2020-09,MO,29,POLK,29167.0,18 to 49 years,Female,NaN,NaN,0.0,0.0,Missing,Missing,Laboratory-confirmed case,Symptomatic,Unknown,Missing,Unknown,NaN
4,2021-11,MO,29,STODDARD,29207.0,65+ years,Female,NaN,NaN,0.0,NaN,Missing,Missing,Laboratory-confirmed case,Missing,Unknown,Missing,Unknown,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
1900007,2023-01,MO,29,NaN,NaN,65+ years,Male,White,Unknown,0.0,NaN,Missing,Missing,Laboratory-confirmed case,Missing,Unknown,Missing,Unknown,NaN
1900008,2023-01,MO,29,NaN,NaN,65+ years,Male,White,Unknown,0.0,NaN,Missing,Missing,Probable Case,Missing,Unknown,Missing,Unknown,NaN
1900009,2023-01,MO,29,NaN,NaN,65+ years,Male,White,Unknown,0.0,NaN,Missing,Missing,Probable Case,Missing,Unknown,Missing,Unknown,NaN
1900010,2023-01,MO,29,NaN,NaN,65+ years,Male,White,Unknown,0.0,NaN,Missing,Missing,Laboratory-confirmed case,Missing,Unknown,Missing,Unknown,NaN


In [2]:
covid_data['death_yn'].value_counts(dropna=False)

death_yn
Unknown    1867098
NaN          22062
Yes           8583
Missing       1602
No             667
Name: count, dtype: int64

In [3]:
mo_county_population = pd.read_csv("co-est2024-pop-29.csv")
mo_county_population

,table with row headers in column A and column headers in rows 3 through 4 (leading dots indicate sub-parts),Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 24,Unnamed: 25,Unnamed: 26,Unnamed: 27,Unnamed: 28,Unnamed: 29,Unnamed: 30,Unnamed: 31,Unnamed: 32,Unnamed: 33
0,Annual Estimates of the Resident Population fo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,Geographic Area,"April 1, 2020 Estimates Base",Population Estimate (as of July 1),NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,NaN,NaN,2020,2021,2022,2023,2024,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,Missouri,"6,154,854","6,154,744","6,171,374","6,179,414","6,208,038","6,245,466",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,".Adair County, Missouri","25,316","25,272","25,188","25,155","25,231","25,660",NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
120,Note: The estimates are developed from a base ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
121,Suggested Citation:,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
122,Annual Estimates of the Resident Population fo...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
123,"Source: U.S. Census Bureau, Population Division",NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [4]:
mo_county_population = pd.read_csv("co-est2024-pop-29.csv", header=2, skipfooter = 6, engine='python')

mo_county_population = mo_county_population.rename(columns={
    'Population Estimate (as of July 1)': '2020 Population Estimate',
    'Unnamed: 3': '2021 Population Estimate',
    'Unnamed: 4': '2022 Population Estimate',
    'Unnamed: 5': '2023 Population Estimate',
    'Unnamed: 6': '2024 Population Estimate'
})

mo_county_population = mo_county_population.drop(0)

mo_county_population['Geographic Area'] = mo_county_population['Geographic Area'].str.replace('.', '', regex=False)
mo_county_population['Geographic Area'] = mo_county_population['Geographic Area'].str.replace(', Missouri', '', regex=False)
mo_county_population['Geographic Area'] = mo_county_population['Geographic Area'].str.replace(' County', '', regex=False).str.upper()
mo_county_population = mo_county_population.loc[:, ~mo_county_population.columns.str.contains('^Unnamed')]
mo_county_population = mo_county_population[mo_county_population['Geographic Area'] != 'MISSOURI']

mo_county_population = mo_county_population.reset_index(drop=True)

mo_county_population

,Geographic Area,"April 1, 2020 Estimates Base",2020 Population Estimate,2021 Population Estimate,2022 Population Estimate,2023 Population Estimate,2024 Population Estimate
0,ADAIR,"25,316","25,272","25,188","25,155","25,231","25,660"
1,ANDREW,"18,130","18,102","18,015","18,008","18,109","18,091"
2,ATCHISON,"5,302","5,305","5,210","5,163","5,116","5,139"
3,AUDRAIN,"24,960","24,781","24,858","24,455","24,342","24,304"
4,BARRY,"34,529","34,542","34,762","34,931","35,310","35,618"
...,...,...,...,...,...,...,...
110,WAYNE,"10,973","10,935","10,925","10,807","10,825","10,820"
111,WEBSTER,"39,084","39,008","39,600","40,369","41,481","42,041"
112,WORTH,"1,973","1,968","1,973","1,943","1,903","1,872"
113,WRIGHT,"18,187","18,214","18,625","19,108","19,373","19,505"


In [5]:
mo_county_GDP = pd.read_csv("MO_Counties-GDP_0.csv")
mo_county_GDP

,Missouri Counties Real and Current GDP,Unnamed: 1,Unnamed: 2,Unnamed: 3,Unnamed: 4,Unnamed: 5,Unnamed: 6,Unnamed: 7,Unnamed: 8,Unnamed: 9,...,Unnamed: 11,Unnamed: 12,Unnamed: 13,Unnamed: 14,Unnamed: 15,Unnamed: 16,Unnamed: 17,Unnamed: 18,Unnamed: 19,Unnamed: 20
0,Location,Real GDP (Inflation Adjusted),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,Current GDP (Not Adjusted for Inflation),NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,NaN,Thousands of chained 2017 US dollars,NaN,NaN,NaN,NaN,Percent change from preceding period,NaN,NaN,NaN,...,Thousands of US dollars,NaN,NaN,NaN,NaN,Percent change from preceding period,NaN,NaN,NaN,NaN
2,NaN,2019,2020,2021,2022,2023,2018-2019,2019-2020,2020-2021,2021-2022,...,2019,2020,2021,2022,2023,2018-2019,2019-2020,2020-2021,2021-2022,2022-2023
3,Missouri,"$321,329,082","$315,332,580","$331,580,860","$339,652,446","$348,487,426",2.3%,-1.9%,5.2%,2.4%,...,"$334,750,152","$335,285,051","$366,441,471","$400,265,064","$430,114,428",4.5%,0.2%,9.3%,9.2%,7.5%
4,"Adair, MO","$807,582","$785,869","$798,824","$818,197","$856,741",-1.6%,-2.7%,1.6%,2.4%,...,"$843,082","$850,303","$900,095","$966,683","$1,066,538",0.8%,0.9%,5.9%,7.4%,10.3%
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
117,"Wright, MO","$326,481","$353,075","$426,532","$435,908","$456,789",-1.8%,8.1%,20.8%,2.2%,...,"$343,396","$382,368","$480,304","$525,808","$577,614",0.8%,11.3%,25.6%,9.5%,9.9%
118,"St. Louis (Independent City), MO","$29,967,598","$28,335,312","$29,840,940","$31,193,060","$31,984,683",1.4%,-5.4%,5.3%,4.5%,...,"$31,278,420","$30,139,332","$32,750,019","$35,788,874","$38,627,738",3.4%,-3.6%,8.7%,9.3%,7.9%
119,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
120,Source: U.S. Bureau of Economic Analysis,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [6]:
mo_county_GDP = pd.read_csv("MO_Counties-GDP_0.csv", header=0, skipfooter = 3, engine='python')

mo_county_GDP_clean = mo_county_GDP.iloc[:, 0:6].copy()
mo_county_GDP_clean = mo_county_GDP_clean.drop([0, 1, 2])

mo_county_GDP_clean.columns = ['County', '2019 Real GDP', '2020 Real GDP', '2021 Real GDP', '2022 Real GDP', '2023 Real GDP']

mo_county_GDP_clean['County'] = mo_county_GDP_clean['County'].str.replace('.', '', regex=False)
mo_county_GDP_clean['County'] = mo_county_GDP_clean['County'].str.replace(', MO', '', regex=False).str.upper()
mo_county_GDP_clean['County'] = mo_county_GDP_clean['County'].str.replace('ST LOUIS (INDEPENDENT CITY)', 'ST LOUIS CITY', regex=False)
mo_county_GDP_clean = mo_county_GDP_clean[mo_county_GDP_clean['County'] != 'MISSOURI']

mo_county_GDP_clean = mo_county_GDP_clean.reset_index(drop=True)

mo_county_GDP_clean

,County,2019 Real GDP,2020 Real GDP,2021 Real GDP,2022 Real GDP,2023 Real GDP
0,ADAIR,"$807,582","$785,869","$798,824","$818,197","$856,741"
1,ANDREW,"$312,660","$318,931","$318,617","$302,785","$308,512"
2,ATCHISON,"$286,286","$316,301","$370,566","$391,949","$389,756"
3,AUDRAIN,"$903,636","$857,671","$884,628","$858,374","$922,972"
4,BARRY,"$1,507,279","$1,452,006","$1,501,693","$1,670,153","$1,588,323"
...,...,...,...,...,...,...
110,WAYNE,"$215,938","$206,803","$214,826","$207,090","$216,356"
111,WEBSTER,"$758,675","$778,276","$867,773","$863,915","$879,885"
112,WORTH,"$57,955","$50,878","$47,247","$43,271","$52,039"
113,WRIGHT,"$326,481","$353,075","$426,532","$435,908","$456,789"


In [7]:
mo_county_avg_household_income = pd.read_csv('HDPulse_data_export.csv', skiprows=3, skipfooter = 8, engine = 'python')

mo_county_avg_household_income = mo_county_avg_household_income.drop([0, 1])
mo_county_avg_household_income = mo_county_avg_household_income.drop(columns=['Rank within US (of 3141 counties)'])

mo_county_avg_household_income = mo_county_avg_household_income.rename(columns={
    'Value (Dollars)': 'Average Household Income'
})
mo_county_avg_household_income['County'] = mo_county_avg_household_income['County'].str.replace(' County', '', regex = False).str.upper()
mo_county_avg_household_income['County'] = mo_county_avg_household_income['County'].str.replace('.', '', regex = False)

mo_county_avg_household_income = mo_county_avg_household_income.reset_index(drop=True)

mo_county_avg_household_income

,County,FIPS,Average Household Income
0,HICKORY,29085,"35,084"
1,PEMISCOT,29155,"40,748"
2,OZARK,29153,"42,329"
3,WAYNE,29223,"43,393"
4,RIPLEY,29181,"43,898"
...,...,...,...
110,LINCOLN,29113,"85,276"
111,CLAY,29047,"86,150"
112,CASS,29037,"87,413"
113,PLATTE,29165,"95,748"


In [8]:
covid_data = covid_data.dropna(subset=['res_county', 'county_fips_code'])
covid_data_2022 = covid_data[covid_data['case_month'].str.startswith('2022')].copy()

In [9]:
covid_infections_by_county_2022 = covid_data_2022.groupby(['res_county', 'county_fips_code']).size().reset_index(name='covid_infections_2022')
covid_infections_by_county_2022['res_county'] = covid_infections_by_county_2022['res_county'].str.replace('.', '', regex=False)
covid_infections_by_county_2022 = covid_infections_by_county_2022.sort_values(by='covid_infections_2022', ascending=False)
covid_infections_by_county_2022 = covid_infections_by_county_2022.reset_index(drop=True)
covid_infections_by_county_2022

,res_county,county_fips_code,covid_infections_2022
0,ST LOUIS,29189.0,124171
1,JACKSON,29095.0,79621
2,ST CHARLES,29183.0,48803
3,GREENE,29077.0,33739
4,ST LOUIS CITY,29510.0,32679
5,CLAY,29047.0,28200
6,JEFFERSON,29099.0,26023
7,BOONE,29019.0,23705
8,JASPER,29097.0,14283
9,CASS,29037.0,11391


In [10]:
deaths_2022 = covid_data_2022[covid_data_2022['death_yn'] == 'Yes']
covid_deaths_by_county_2022 = deaths_2022.groupby(['res_county', 'county_fips_code']).size().reset_index(name='covid_deaths_2022')
covid_deaths_by_county_2022['res_county'] = covid_deaths_by_county_2022['res_county'].str.replace('.', '', regex=False)
covid_infections_by_county_2022 = covid_infections_by_county_2022.merge(covid_deaths_by_county_2022, on=['res_county', 'county_fips_code'], how='left')
covid_infections_by_county_2022['covid_deaths_2022'] = covid_infections_by_county_2022['covid_deaths_2022'].fillna(0).astype(int)
covid_infections_by_county_2022

,res_county,county_fips_code,covid_infections_2022,covid_deaths_2022
0,ST LOUIS,29189.0,124171,358
1,JACKSON,29095.0,79621,207
2,ST CHARLES,29183.0,48803,49
3,GREENE,29077.0,33739,75
4,ST LOUIS CITY,29510.0,32679,31
5,CLAY,29047.0,28200,48
6,JEFFERSON,29099.0,26023,62
7,BOONE,29019.0,23705,0
8,JASPER,29097.0,14283,31
9,CASS,29037.0,11391,14


In [11]:
mo_county_population_2022 = mo_county_population[['Geographic Area','2022 Population Estimate']].copy()
mo_county_population_2022

,Geographic Area,2022 Population Estimate
0,ADAIR,"25,155"
1,ANDREW,"18,008"
2,ATCHISON,"5,163"
3,AUDRAIN,"24,455"
4,BARRY,"34,931"
...,...,...
110,WAYNE,"10,807"
111,WEBSTER,"40,369"
112,WORTH,"1,943"
113,WRIGHT,"19,108"


In [12]:
mo_county_GDP_clean_2022 = mo_county_GDP_clean[['County', '2022 Real GDP']].copy()
mo_county_GDP_clean_2022

,County,2022 Real GDP
0,ADAIR,"$818,197"
1,ANDREW,"$302,785"
2,ATCHISON,"$391,949"
3,AUDRAIN,"$858,374"
4,BARRY,"$1,670,153"
...,...,...
110,WAYNE,"$207,090"
111,WEBSTER,"$863,915"
112,WORTH,"$43,271"
113,WRIGHT,"$435,908"


In [13]:
covid_infections_by_county_2022 = covid_infections_by_county_2022.rename(columns={'res_county': 'County'})
mo_county_population_2022 = mo_county_population_2022.rename(columns={'Geographic Area': 'County'})

merged_data = covid_infections_by_county_2022.merge(mo_county_population_2022, on='County') \
    .merge(mo_county_GDP_clean_2022, on='County') \
    .merge(mo_county_avg_household_income, on='County')

merged_data = merged_data.drop(columns=['FIPS'])

merged_data

,County,county_fips_code,covid_infections_2022,covid_deaths_2022,2022 Population Estimate,2022 Real GDP,Average Household Income
0,ST LOUIS,29189.0,124171,358,"991,881","$89,147,507","81,340"
1,JACKSON,29095.0,79621,207,"716,580","$50,118,029","67,178"
2,ST CHARLES,29183.0,48803,49,"414,055","$19,607,818","102,912"
3,GREENE,29077.0,33739,75,"303,336","$18,163,171","57,488"
4,ST LOUIS CITY,29510.0,32679,31,"286,292","$31,193,060","55,279"
5,CLAY,29047.0,28200,48,"257,037","$13,662,294","86,150"
6,JEFFERSON,29099.0,26023,62,"229,268","$5,241,427","80,522"
7,BOONE,29019.0,23705,0,"187,743","$9,997,270","69,913"
8,JASPER,29097.0,14283,31,"123,969","$5,475,966","57,525"
9,CASS,29037.0,11391,14,"110,345","$2,962,114","87,413"


In [14]:
merged_data['2022 Population Estimate'] = merged_data['2022 Population Estimate'].str.replace(',', '', regex = True).astype(int)
merged_data['2022 Real GDP'] = merged_data['2022 Real GDP'].str.replace(r'[\$,]', '', regex = True).astype(int)
merged_data['Average Household Income'] = merged_data['Average Household Income'].str.replace(',', '', regex = True).astype(int)

merged_data['Covid Infections Per Capita'] = merged_data['covid_infections_2022']/merged_data['2022 Population Estimate']
merged_data['Covid Deaths Per Capita'] = merged_data['covid_deaths_2022']/merged_data['2022 Population Estimate']

In [15]:
merged_data

,County,county_fips_code,covid_infections_2022,covid_deaths_2022,2022 Population Estimate,2022 Real GDP,Average Household Income,Covid Infections Per Capita,Covid Deaths Per Capita
0,ST LOUIS,29189.0,124171,358,991881,89147507,81340,0.125187,0.000361
1,JACKSON,29095.0,79621,207,716580,50118029,67178,0.111113,0.000289
2,ST CHARLES,29183.0,48803,49,414055,19607818,102912,0.117866,0.000118
3,GREENE,29077.0,33739,75,303336,18163171,57488,0.111226,0.000247
4,ST LOUIS CITY,29510.0,32679,31,286292,31193060,55279,0.114146,0.000108
5,CLAY,29047.0,28200,48,257037,13662294,86150,0.109712,0.000187
6,JEFFERSON,29099.0,26023,62,229268,5241427,80522,0.113505,0.000270
7,BOONE,29019.0,23705,0,187743,9997270,69913,0.126263,0.000000
8,JASPER,29097.0,14283,31,123969,5475966,57525,0.115214,0.000250
9,CASS,29037.0,11391,14,110345,2962114,87413,0.103231,0.000127


In [16]:
merged_data = merged_data.rename(columns={
    'county_fips_code': 'County FIPS Code',
    'covid_infections_2022': '2022 Covid Infections',
    'covid_deaths_2022': '2022 Covid Deaths'
})

merged_data

,County,County FIPS Code,2022 Covid Infections,2022 Covid Deaths,2022 Population Estimate,2022 Real GDP,Average Household Income,Covid Infections Per Capita,Covid Deaths Per Capita
0,ST LOUIS,29189.0,124171,358,991881,89147507,81340,0.125187,0.000361
1,JACKSON,29095.0,79621,207,716580,50118029,67178,0.111113,0.000289
2,ST CHARLES,29183.0,48803,49,414055,19607818,102912,0.117866,0.000118
3,GREENE,29077.0,33739,75,303336,18163171,57488,0.111226,0.000247
4,ST LOUIS CITY,29510.0,32679,31,286292,31193060,55279,0.114146,0.000108
5,CLAY,29047.0,28200,48,257037,13662294,86150,0.109712,0.000187
6,JEFFERSON,29099.0,26023,62,229268,5241427,80522,0.113505,0.000270
7,BOONE,29019.0,23705,0,187743,9997270,69913,0.126263,0.000000
8,JASPER,29097.0,14283,31,123969,5475966,57525,0.115214,0.000250
9,CASS,29037.0,11391,14,110345,2962114,87413,0.103231,0.000127


In [17]:
merged_data.to_csv('missouri_counties_covid_and_economic_data_2022.csv', index=False)